# 02 - Synthetic Generation

Check the local Ollama Qwen3-4B setup, generate one English AG News-style article, and run a basic English-language smoke check.

In [1]:
import shutil
import requests

MODEL = "qwen3:4b-instruct"
API = "http://127.0.0.1:11434"

assert shutil.which("ollama"), "Install Ollama and add it to PATH."
response = requests.get(f"{API}/api/tags", timeout=10)
response.raise_for_status()
installed = next((item for item in response.json()["models"] if item["name"] == MODEL), None)
assert installed, f"Download the model with: ollama pull {MODEL}"
assert installed["details"]["family"] == "qwen3"
assert installed["details"]["parameter_size"] == "4.0B"
print({"model": MODEL, "digest": installed["digest"],
       "quantization": installed["details"]["quantization_level"]})

{'model': 'qwen3:4b-instruct', 'digest': '0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0', 'quantization': 'Q4_K_M'}


In [2]:
import re

prompt = (
    "Write one original, plausible but fictional AG News-style article in English "
    "for the World category. Return exactly two lines: Headline: <headline> "
    "and Summary: <two or three sentences>."
)
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": prompt}],
    "stream": False,
    "think": False,
    "options": {"seed": 42, "temperature": 0.7, "top_p": 0.8,
                "top_k": 20, "num_predict": 160},
}
response = requests.post(f"{API}/api/chat", json=payload, timeout=180)
response.raise_for_status()
article = response.json()["message"]["content"].strip()
print(article)

assert re.search(r"^Headline:\s*.+\nSummary:\s*.+", article, re.DOTALL)
words = re.findall(r"[A-Za-z]+", article.lower())
common_english = {"the", "and", "of", "to", "in", "for", "on", "with", "from", "that"}
assert len(words) >= 30 and sum(word in common_english for word in words) >= 4,     "English-language smoke check failed. Inspect the generated article."
print("English-language smoke check passed.")

Headline: Global Climate Summit Unveils Historic Agreement to Limit Carbon Emissions by 2030  
Summary: Leaders from 195 nations reached a landmark accord during the 2024 Global Climate Summit in Dubai, committing to cut global carbon emissions by 50% by 2030. The agreement includes binding targets, financial pledges, and expanded support for developing nations to transition to renewable energy. Experts say it marks the most comprehensive climate action plan in decades.
English-language smoke check passed.


The generated article is a fictional test sample; this check does not verify its factual accuracy.